In [1]:
!wget https://raw.githubusercontent.com/apache/spark/refs/heads/master/data/mllib/als/sample_movielens_ratings.txt

--2026-03-15 00:35:46--  https://raw.githubusercontent.com/apache/spark/refs/heads/master/data/mllib/als/sample_movielens_ratings.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32363 (32K) [text/plain]
Saving to: ‘sample_movielens_ratings.txt’

sample_movielens_ra 100%[===================>]  31.60K  --.-KB/s    in 0s      

2026-03-15 00:35:47 (112 MB/s) - ‘sample_movielens_ratings.txt’ saved [32363/32363]



In [4]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('movie-len').getOrCreate()

lines = spark.read.text("sample_movielens_ratings.txt").rdd
parts = lines.map(lambda row: row.value.split("::"))
ratingsRDD = parts.map(lambda p: Row(userId=int(p[0]), movieId=int(p[1]),
                                     rating=float(p[2]), timestamp=int(p[3])))
ratings = spark.createDataFrame(ratingsRDD)
(training, test) = ratings.randomSplit([0.8, 0.2])

# Build the recommendation model using ALS on the training data
# Note we set cold start strategy to 'drop' to ensure we don't get NaN evaluation metrics
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(training)

# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

# Generate top 10 movie recommendations for each user
userRecs = model.recommendForAllUsers(10)
# Generate top 10 user recommendations for each movie
movieRecs = model.recommendForAllItems(10)

# Generate top 10 movie recommendations for a specified set of users
users = ratings.select(als.getUserCol()).distinct().limit(3)
userSubsetRecs = model.recommendForUserSubset(users, 10)
# Generate top 10 user recommendations for a specified set of movies
movies = ratings.select(als.getItemCol()).distinct().limit(3)
movieSubSetRecs = model.recommendForItemSubset(movies, 10)

26/03/15 00:37:40 WARN Utils: Your hostname, bigdata resolves to a loopback address: 127.0.1.1; using 10.3.135.140 instead (on interface ens3)
26/03/15 00:37:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/15 00:37:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/15 00:37:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/15 00:37:41 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/15 00:37:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/03/15 00:37:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
26/03/1

Root-mean-square error = 1.850430188271468
